# CS 370 Project Two - Pirate Intelligent Agent<a href="#CS-370-Project-Two---Pirate-Intelligent-Agent"
class="anchor-link" rel="noopener">¶</a>

This notebook completes the deep Q-learning training algorithm for the
pirate agent.

In \[1\]:

    from __future__ import print_function
    import os, sys, time, datetime, json, random
    import numpy as np

    try:
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import Dense, PReLU
        from tensorflow.keras.optimizers import SGD, Adam, RMSprop
    except ImportError:
        from keras.models import Sequential
        from keras.layers import Dense, PReLU
        from keras.optimizers import SGD, Adam, RMSprop

    import matplotlib.pyplot as plt
    from TreasureMaze import TreasureMaze
    from GameExperience import GameExperience

    %matplotlib inline

## Maze environment<a href="#Maze-environment" class="anchor-link" rel="noopener">¶</a>

In \[2\]:

    maze = np.array([
        [1., 0., 1., 1., 1., 1., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 1., 1., 1., 0., 1., 0., 1.],
        [1., 1., 1., 0., 1., 1., 1., 1.],
        [1., 1., 0., 1., 1., 1., 1., 1.],
        [1., 1., 1., 0., 1., 0., 0., 0.],
        [1., 1., 1., 0., 1., 1., 1., 1.],
        [1., 1., 1., 1., 0., 1., 1., 1.]
    ])

In \[3\]:

    def show(qmaze):
        plt.grid('on')
        nrows, ncols = qmaze.maze.shape
        ax = plt.gca()
        ax.set_xticks(np.arange(0.5, nrows, 1))
        ax.set_yticks(np.arange(0.5, ncols, 1))
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        canvas = np.copy(qmaze.maze)
        for row, col in qmaze.visited:
            canvas[row, col] = 0.6
        pirate_row, pirate_col, _ = qmaze.state
        canvas[pirate_row, pirate_col] = 0.3
        canvas[nrows - 1, ncols - 1] = 0.9
        img = plt.imshow(canvas, interpolation='none', cmap='gray')
        return img

    LEFT = 0
    UP = 1
    RIGHT = 2
    DOWN = 3

    epsilon = 0.1

    actions_dict = {
        LEFT: 'left',
        UP: 'up',
        RIGHT: 'right',
        DOWN: 'down',
    }
    num_actions = len(actions_dict)

## Helper functions<a href="#Helper-functions" class="anchor-link" rel="noopener">¶</a>

In \[4\]:

    def format_time(seconds):
        if seconds < 60:
            return '%.1f seconds' % seconds
        elif seconds < 3600:
            return '%.2f minutes' % (seconds / 60.0)
        return '%.2f hours' % (seconds / 3600.0)


    def play_game(model, qmaze, pirate_cell):
        qmaze.reset(pirate_cell)
        envstate = qmaze.observe()
        while True:
            prev_envstate = envstate
            q = model.predict(prev_envstate, verbose=0)
            action = np.argmax(q[0])
            envstate, reward, game_status = qmaze.act(action)
            if game_status == 'win':
                return True
            elif game_status == 'lose':
                return False


    def completion_check(model, qmaze):
        for cell in qmaze.free_cells:
            if not qmaze.valid_actions(cell):
                return False
            if not play_game(model, qmaze, cell):
                return False
        return True

## Build the neural network model<a href="#Build-the-neural-network-model" class="anchor-link"
rel="noopener">¶</a>

In \[5\]:

    def build_model(maze):
        model = Sequential()
        model.add(Dense(maze.size, input_shape=(maze.size,)))
        model.add(PReLU())
        model.add(Dense(maze.size))
        model.add(PReLU())
        model.add(Dense(num_actions))
        model.compile(optimizer='adam', loss='mse')
        return model

## Q-training algorithm<a href="#Q-training-algorithm" class="anchor-link" rel="noopener">¶</a>

In \[6\]:

    def qtrain(model, maze, **opt):
        global epsilon

        n_epoch = opt.get('n_epoch', opt.get('epochs', 15000))
        max_memory = opt.get('max_memory', 1000)
        data_size = opt.get('data_size', 50)
        sync_freq = opt.get('sync_freq', 20)

        start_time = datetime.datetime.now()
        qmaze = TreasureMaze(maze)

        target_model = build_model(maze)
        target_model.set_weights(model.get_weights())

        experience = GameExperience(model, target_model, max_memory=max_memory)

        win_history = []
        hsize = qmaze.maze.size // 2
        win_rate = 0.0

        for epoch in range(n_epoch):
            loss = 0.0
            pirate_cell = random.choice(qmaze.free_cells)
            qmaze.reset(pirate_cell)
            envstate = qmaze.observe()
            game_status = 'not_over'
            n_episodes = 0

            while game_status == 'not_over':
                prev_envstate = envstate

                valid_actions = qmaze.valid_actions()
                if np.random.rand() < epsilon:
                    action = random.choice(valid_actions)
                else:
                    q = model.predict(prev_envstate, verbose=0)[0]
                    valid_q = [(a, q[a]) for a in valid_actions]
                    action = max(valid_q, key=lambda item: item[1])[0]

                envstate, reward, game_status = qmaze.act(action)
                done = (game_status != 'not_over')

                episode = [prev_envstate, action, reward, envstate, done]
                experience.remember(episode)
                n_episodes += 1

                if len(experience.memory) >= data_size:
                    inputs, targets = experience.get_data(data_size)
                    history = model.fit(inputs, targets, epochs=1, batch_size=16, verbose=0)
                    loss = history.history['loss'][0]

            if epoch % sync_freq == 0:
                target_model.set_weights(model.get_weights())

            win_history.append(1 if game_status == 'win' else 0)
            if len(win_history) > hsize:
                win_history = win_history[-hsize:]
            win_rate = sum(win_history) / float(len(win_history))

            dt = datetime.datetime.now() - start_time
            t = format_time(dt.total_seconds())
            template = "Epoch: {:03d}/{:d} | Loss: {:.4f} | Episodes: {:d} | Win count: {:d} | Win rate: {:.3f} | time: {}"
            print(template.format(epoch, n_epoch - 1, loss, n_episodes, sum(win_history), win_rate, t))

            if win_rate > 0.9:
                epsilon = 0.05

            if len(win_history) >= hsize and sum(win_history[-hsize:]) == hsize and completion_check(model, qmaze):
                print("Reached 100% win rate at epoch: %d" % epoch)
                break

        dt = datetime.datetime.now() - start_time
        seconds = dt.total_seconds()
        print("n_epoch: %d, max_memory: %d, data_size: %d, time: %s" %
              (epoch + 1, max_memory, data_size, format_time(seconds)))
        return seconds

## Train the model<a href="#Train-the-model" class="anchor-link" rel="noopener">¶</a>

In \[7\]:

    epsilon = 0.1
    model = build_model(maze)
    training_time = qtrain(model, maze, epochs=300, max_memory=8 * maze.size, data_size=32)

    /home/codio/.pyenv/versions/3.11.9/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
      super().__init__(activity_regularizer=activity_regularizer, **kwargs)
    WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
    I0000 00:00:1776792821.765307  932211 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
    I0000 00:00:1776792821.818945  932211 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
    I0000 00:00:1776792821.820458  932211 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
    I0000 00:00:1776792821.822798  932211 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
    I0000 00:00:1776792821.824225  932211 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
    I0000 00:00:1776792821.825601  932211 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
    I0000 00:00:1776792822.016301  932211 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
    I0000 00:00:1776792822.017874  932211 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
    I0000 00:00:1776792822.019263  932211 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
    WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
    I0000 00:00:1776792822.310755  932294 service.cc:146] XLA service 0x75b754005c60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
    I0000 00:00:1776792822.310794  932294 service.cc:154]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
    I0000 00:00:1776792822.442310  932294 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.

    Epoch: 000/299 | Loss: 0.0011 | Episodes: 139 | Win count: 0 | Win rate: 0.000 | time: 15.0 seconds
    Epoch: 001/299 | Loss: 0.0152 | Episodes: 31 | Win count: 1 | Win rate: 0.500 | time: 18.2 seconds
    Epoch: 002/299 | Loss: 0.0008 | Episodes: 145 | Win count: 1 | Win rate: 0.333 | time: 34.3 seconds
    Epoch: 003/299 | Loss: 0.0013 | Episodes: 144 | Win count: 1 | Win rate: 0.250 | time: 50.0 seconds
    Epoch: 004/299 | Loss: 0.0014 | Episodes: 143 | Win count: 1 | Win rate: 0.200 | time: 1.10 minutes
    Epoch: 005/299 | Loss: 0.0009 | Episodes: 144 | Win count: 1 | Win rate: 0.167 | time: 1.36 minutes
    Epoch: 006/299 | Loss: 0.0002 | Episodes: 134 | Win count: 1 | Win rate: 0.143 | time: 1.60 minutes
    Epoch: 007/299 | Loss: 0.0006 | Episodes: 138 | Win count: 1 | Win rate: 0.125 | time: 1.85 minutes
    Epoch: 008/299 | Loss: 0.0006 | Episodes: 150 | Win count: 1 | Win rate: 0.111 | time: 2.13 minutes
    Epoch: 009/299 | Loss: 0.0099 | Episodes: 85 | Win count: 2 | Win rate: 0.200 | time: 2.29 minutes
    Epoch: 010/299 | Loss: 0.0011 | Episodes: 26 | Win count: 3 | Win rate: 0.273 | time: 2.33 minutes
    Epoch: 011/299 | Loss: 0.0008 | Episodes: 7 | Win count: 4 | Win rate: 0.333 | time: 2.35 minutes
    Epoch: 012/299 | Loss: 0.0011 | Episodes: 50 | Win count: 5 | Win rate: 0.385 | time: 2.44 minutes
    Epoch: 013/299 | Loss: 0.0011 | Episodes: 134 | Win count: 5 | Win rate: 0.357 | time: 2.69 minutes
    Epoch: 014/299 | Loss: 0.0025 | Episodes: 36 | Win count: 6 | Win rate: 0.400 | time: 2.75 minutes
    Epoch: 015/299 | Loss: 0.0005 | Episodes: 133 | Win count: 6 | Win rate: 0.375 | time: 3.00 minutes
    Epoch: 016/299 | Loss: 0.0016 | Episodes: 28 | Win count: 7 | Win rate: 0.412 | time: 3.05 minutes
    Epoch: 017/299 | Loss: 0.0010 | Episodes: 4 | Win count: 8 | Win rate: 0.444 | time: 3.06 minutes
    Epoch: 018/299 | Loss: 0.0011 | Episodes: 4 | Win count: 9 | Win rate: 0.474 | time: 3.06 minutes
    Epoch: 019/299 | Loss: 0.0012 | Episodes: 9 | Win count: 10 | Win rate: 0.500 | time: 3.08 minutes
    Epoch: 020/299 | Loss: 0.0009 | Episodes: 84 | Win count: 11 | Win rate: 0.524 | time: 3.24 minutes
    Epoch: 021/299 | Loss: 0.0117 | Episodes: 7 | Win count: 12 | Win rate: 0.545 | time: 3.25 minutes
    Epoch: 022/299 | Loss: 0.0063 | Episodes: 32 | Win count: 13 | Win rate: 0.565 | time: 3.31 minutes
    Epoch: 023/299 | Loss: 0.0018 | Episodes: 135 | Win count: 13 | Win rate: 0.542 | time: 3.55 minutes
    Epoch: 024/299 | Loss: 0.0012 | Episodes: 23 | Win count: 14 | Win rate: 0.560 | time: 3.59 minutes
    Epoch: 025/299 | Loss: 0.0094 | Episodes: 136 | Win count: 14 | Win rate: 0.538 | time: 3.85 minutes
    Epoch: 026/299 | Loss: 0.0025 | Episodes: 13 | Win count: 15 | Win rate: 0.556 | time: 3.87 minutes
    Epoch: 027/299 | Loss: 0.0021 | Episodes: 79 | Win count: 16 | Win rate: 0.571 | time: 4.01 minutes
    Epoch: 028/299 | Loss: 0.0011 | Episodes: 6 | Win count: 17 | Win rate: 0.586 | time: 4.02 minutes
    Epoch: 029/299 | Loss: 0.0034 | Episodes: 15 | Win count: 18 | Win rate: 0.600 | time: 4.05 minutes
    Epoch: 030/299 | Loss: 0.0022 | Episodes: 22 | Win count: 19 | Win rate: 0.613 | time: 4.09 minutes
    Epoch: 031/299 | Loss: 0.0142 | Episodes: 30 | Win count: 20 | Win rate: 0.625 | time: 4.14 minutes
    Epoch: 032/299 | Loss: 0.0019 | Episodes: 36 | Win count: 21 | Win rate: 0.656 | time: 4.21 minutes
    Epoch: 033/299 | Loss: 0.0018 | Episodes: 77 | Win count: 21 | Win rate: 0.656 | time: 4.35 minutes
    Epoch: 034/299 | Loss: 0.0039 | Episodes: 14 | Win count: 22 | Win rate: 0.688 | time: 4.38 minutes
    Epoch: 035/299 | Loss: 0.0023 | Episodes: 17 | Win count: 23 | Win rate: 0.719 | time: 4.41 minutes
    Epoch: 036/299 | Loss: 0.0014 | Episodes: 9 | Win count: 24 | Win rate: 0.750 | time: 4.43 minutes
    Epoch: 037/299 | Loss: 0.0036 | Episodes: 30 | Win count: 25 | Win rate: 0.781 | time: 4.48 minutes
    Epoch: 038/299 | Loss: 0.0021 | Episodes: 5 | Win count: 26 | Win rate: 0.812 | time: 4.49 minutes
    Epoch: 039/299 | Loss: 0.0057 | Episodes: 46 | Win count: 27 | Win rate: 0.844 | time: 4.57 minutes
    Epoch: 040/299 | Loss: 0.0069 | Episodes: 36 | Win count: 28 | Win rate: 0.875 | time: 4.64 minutes
    Epoch: 041/299 | Loss: 0.0097 | Episodes: 4 | Win count: 28 | Win rate: 0.875 | time: 4.65 minutes
    Epoch: 042/299 | Loss: 0.0084 | Episodes: 8 | Win count: 28 | Win rate: 0.875 | time: 4.66 minutes
    Epoch: 043/299 | Loss: 0.0051 | Episodes: 150 | Win count: 27 | Win rate: 0.844 | time: 4.94 minutes
    Epoch: 044/299 | Loss: 0.0033 | Episodes: 76 | Win count: 27 | Win rate: 0.844 | time: 5.08 minutes
    Epoch: 045/299 | Loss: 0.0023 | Episodes: 42 | Win count: 28 | Win rate: 0.875 | time: 5.16 minutes
    Epoch: 046/299 | Loss: 0.0029 | Episodes: 12 | Win count: 28 | Win rate: 0.875 | time: 5.19 minutes
    Epoch: 047/299 | Loss: 0.0014 | Episodes: 57 | Win count: 29 | Win rate: 0.906 | time: 5.29 minutes
    Epoch: 048/299 | Loss: 0.0008 | Episodes: 46 | Win count: 29 | Win rate: 0.906 | time: 5.38 minutes
    Epoch: 049/299 | Loss: 0.0013 | Episodes: 12 | Win count: 29 | Win rate: 0.906 | time: 5.41 minutes
    Epoch: 050/299 | Loss: 0.0009 | Episodes: 32 | Win count: 29 | Win rate: 0.906 | time: 5.47 minutes
    Epoch: 051/299 | Loss: 0.0023 | Episodes: 7 | Win count: 29 | Win rate: 0.906 | time: 5.48 minutes
    Epoch: 052/299 | Loss: 0.0013 | Episodes: 30 | Win count: 29 | Win rate: 0.906 | time: 5.54 minutes
    Epoch: 053/299 | Loss: 0.0012 | Episodes: 3 | Win count: 29 | Win rate: 0.906 | time: 5.54 minutes
    Epoch: 054/299 | Loss: 0.0073 | Episodes: 2 | Win count: 29 | Win rate: 0.906 | time: 5.55 minutes
    Epoch: 055/299 | Loss: 0.0007 | Episodes: 23 | Win count: 30 | Win rate: 0.938 | time: 5.59 minutes
    Epoch: 056/299 | Loss: 0.0035 | Episodes: 46 | Win count: 30 | Win rate: 0.938 | time: 5.68 minutes
    Epoch: 057/299 | Loss: 0.0010 | Episodes: 44 | Win count: 31 | Win rate: 0.969 | time: 5.76 minutes
    Epoch: 058/299 | Loss: 0.0024 | Episodes: 56 | Win count: 31 | Win rate: 0.969 | time: 5.86 minutes
    Epoch: 059/299 | Loss: 0.0020 | Episodes: 9 | Win count: 31 | Win rate: 0.969 | time: 5.88 minutes
    Epoch: 060/299 | Loss: 0.0011 | Episodes: 32 | Win count: 31 | Win rate: 0.969 | time: 5.94 minutes
    Epoch: 061/299 | Loss: 0.0024 | Episodes: 133 | Win count: 30 | Win rate: 0.938 | time: 6.19 minutes
    Epoch: 062/299 | Loss: 0.0014 | Episodes: 131 | Win count: 29 | Win rate: 0.906 | time: 6.44 minutes
    Epoch: 063/299 | Loss: 0.0003 | Episodes: 142 | Win count: 28 | Win rate: 0.875 | time: 6.72 minutes
    Epoch: 064/299 | Loss: 0.0013 | Episodes: 141 | Win count: 27 | Win rate: 0.844 | time: 6.99 minutes
    Epoch: 065/299 | Loss: 0.0003 | Episodes: 83 | Win count: 27 | Win rate: 0.844 | time: 7.14 minutes
    Epoch: 066/299 | Loss: 0.0004 | Episodes: 27 | Win count: 27 | Win rate: 0.844 | time: 7.19 minutes
    Epoch: 067/299 | Loss: 0.0007 | Episodes: 6 | Win count: 27 | Win rate: 0.844 | time: 7.21 minutes
    Epoch: 068/299 | Loss: 0.0006 | Episodes: 40 | Win count: 27 | Win rate: 0.844 | time: 7.28 minutes
    Epoch: 069/299 | Loss: 0.0007 | Episodes: 120 | Win count: 27 | Win rate: 0.844 | time: 7.51 minutes
    Epoch: 070/299 | Loss: 0.0010 | Episodes: 138 | Win count: 26 | Win rate: 0.812 | time: 7.77 minutes
    Epoch: 071/299 | Loss: 0.0004 | Episodes: 123 | Win count: 26 | Win rate: 0.812 | time: 8.00 minutes
    Epoch: 072/299 | Loss: 0.0002 | Episodes: 38 | Win count: 26 | Win rate: 0.812 | time: 8.07 minutes
    Epoch: 073/299 | Loss: 0.0004 | Episodes: 2 | Win count: 26 | Win rate: 0.812 | time: 8.08 minutes
    Epoch: 074/299 | Loss: 0.0003 | Episodes: 18 | Win count: 26 | Win rate: 0.812 | time: 8.11 minutes
    Epoch: 075/299 | Loss: 0.0004 | Episodes: 142 | Win count: 26 | Win rate: 0.812 | time: 8.38 minutes
    Epoch: 076/299 | Loss: 0.0003 | Episodes: 42 | Win count: 26 | Win rate: 0.812 | time: 8.46 minutes
    Epoch: 077/299 | Loss: 0.0004 | Episodes: 29 | Win count: 26 | Win rate: 0.812 | time: 8.52 minutes
    Epoch: 078/299 | Loss: 0.0005 | Episodes: 3 | Win count: 26 | Win rate: 0.812 | time: 8.52 minutes
    Epoch: 079/299 | Loss: 0.0006 | Episodes: 1 | Win count: 26 | Win rate: 0.812 | time: 8.52 minutes
    Epoch: 080/299 | Loss: 0.0003 | Episodes: 143 | Win count: 25 | Win rate: 0.781 | time: 8.80 minutes
    Epoch: 081/299 | Loss: 0.0034 | Episodes: 142 | Win count: 24 | Win rate: 0.750 | time: 9.07 minutes
    Epoch: 082/299 | Loss: 0.0035 | Episodes: 135 | Win count: 24 | Win rate: 0.750 | time: 9.32 minutes
    Epoch: 083/299 | Loss: 0.0026 | Episodes: 136 | Win count: 23 | Win rate: 0.719 | time: 9.58 minutes
    Epoch: 084/299 | Loss: 0.0009 | Episodes: 8 | Win count: 23 | Win rate: 0.719 | time: 9.59 minutes
    Epoch: 085/299 | Loss: 0.0005 | Episodes: 7 | Win count: 23 | Win rate: 0.719 | time: 9.61 minutes
    Epoch: 086/299 | Loss: 0.0024 | Episodes: 135 | Win count: 22 | Win rate: 0.688 | time: 9.88 minutes
    Epoch: 087/299 | Loss: 0.0006 | Episodes: 114 | Win count: 22 | Win rate: 0.688 | time: 10.10 minutes
    Epoch: 088/299 | Loss: 0.0015 | Episodes: 1 | Win count: 22 | Win rate: 0.688 | time: 10.10 minutes
    Epoch: 089/299 | Loss: 0.0016 | Episodes: 11 | Win count: 22 | Win rate: 0.688 | time: 10.12 minutes
    Epoch: 090/299 | Loss: 0.0011 | Episodes: 7 | Win count: 22 | Win rate: 0.688 | time: 10.13 minutes
    Epoch: 091/299 | Loss: 0.0013 | Episodes: 2 | Win count: 22 | Win rate: 0.688 | time: 10.14 minutes
    Epoch: 092/299 | Loss: 0.0016 | Episodes: 137 | Win count: 21 | Win rate: 0.656 | time: 10.40 minutes
    Epoch: 093/299 | Loss: 0.0012 | Episodes: 1 | Win count: 22 | Win rate: 0.688 | time: 10.40 minutes
    Epoch: 094/299 | Loss: 0.0026 | Episodes: 139 | Win count: 22 | Win rate: 0.688 | time: 10.67 minutes
    Epoch: 095/299 | Loss: 0.0041 | Episodes: 22 | Win count: 23 | Win rate: 0.719 | time: 10.71 minutes
    Epoch: 096/299 | Loss: 0.0024 | Episodes: 77 | Win count: 24 | Win rate: 0.750 | time: 10.86 minutes
    Epoch: 097/299 | Loss: 0.0024 | Episodes: 7 | Win count: 24 | Win rate: 0.750 | time: 10.87 minutes
    Epoch: 098/299 | Loss: 0.0027 | Episodes: 8 | Win count: 24 | Win rate: 0.750 | time: 10.89 minutes
    Epoch: 099/299 | Loss: 0.0018 | Episodes: 133 | Win count: 23 | Win rate: 0.719 | time: 11.14 minutes
    Epoch: 100/299 | Loss: 0.0019 | Episodes: 19 | Win count: 23 | Win rate: 0.719 | time: 11.17 minutes
    Epoch: 101/299 | Loss: 0.0029 | Episodes: 1 | Win count: 23 | Win rate: 0.719 | time: 11.18 minutes
    Epoch: 102/299 | Loss: 0.0047 | Episodes: 52 | Win count: 24 | Win rate: 0.750 | time: 11.28 minutes
    Epoch: 103/299 | Loss: 0.0034 | Episodes: 1 | Win count: 24 | Win rate: 0.750 | time: 11.28 minutes
    Epoch: 104/299 | Loss: 0.0021 | Episodes: 23 | Win count: 24 | Win rate: 0.750 | time: 11.32 minutes
    Epoch: 105/299 | Loss: 0.0030 | Episodes: 138 | Win count: 23 | Win rate: 0.719 | time: 11.59 minutes
    Epoch: 106/299 | Loss: 0.0014 | Episodes: 11 | Win count: 23 | Win rate: 0.719 | time: 11.61 minutes
    Epoch: 107/299 | Loss: 0.0020 | Episodes: 21 | Win count: 24 | Win rate: 0.750 | time: 11.65 minutes
    Epoch: 108/299 | Loss: 0.0022 | Episodes: 16 | Win count: 24 | Win rate: 0.750 | time: 11.68 minutes
    Epoch: 109/299 | Loss: 0.0056 | Episodes: 26 | Win count: 24 | Win rate: 0.750 | time: 11.73 minutes
    Epoch: 110/299 | Loss: 0.0028 | Episodes: 7 | Win count: 24 | Win rate: 0.750 | time: 11.75 minutes
    Epoch: 111/299 | Loss: 0.0008 | Episodes: 101 | Win count: 24 | Win rate: 0.750 | time: 11.94 minutes
    Epoch: 112/299 | Loss: 0.0053 | Episodes: 28 | Win count: 25 | Win rate: 0.781 | time: 12.00 minutes
    Epoch: 113/299 | Loss: 0.0017 | Episodes: 21 | Win count: 26 | Win rate: 0.812 | time: 12.04 minutes
    Epoch: 114/299 | Loss: 0.0014 | Episodes: 4 | Win count: 26 | Win rate: 0.812 | time: 12.04 minutes
    Epoch: 115/299 | Loss: 0.0009 | Episodes: 16 | Win count: 27 | Win rate: 0.844 | time: 12.07 minutes
    Epoch: 116/299 | Loss: 0.0013 | Episodes: 3 | Win count: 27 | Win rate: 0.844 | time: 12.08 minutes
    Epoch: 117/299 | Loss: 0.0013 | Episodes: 27 | Win count: 27 | Win rate: 0.844 | time: 12.13 minutes
    Epoch: 118/299 | Loss: 0.0011 | Episodes: 21 | Win count: 28 | Win rate: 0.875 | time: 12.17 minutes
    Epoch: 119/299 | Loss: 0.0010 | Episodes: 28 | Win count: 28 | Win rate: 0.875 | time: 12.22 minutes
    Epoch: 120/299 | Loss: 0.0033 | Episodes: 2 | Win count: 28 | Win rate: 0.875 | time: 12.22 minutes
    Epoch: 121/299 | Loss: 0.0074 | Episodes: 1 | Win count: 28 | Win rate: 0.875 | time: 12.23 minutes
    Epoch: 122/299 | Loss: 0.0029 | Episodes: 97 | Win count: 28 | Win rate: 0.875 | time: 12.41 minutes
    Epoch: 123/299 | Loss: 0.0030 | Episodes: 52 | Win count: 28 | Win rate: 0.875 | time: 12.51 minutes
    Epoch: 124/299 | Loss: 0.0015 | Episodes: 134 | Win count: 28 | Win rate: 0.875 | time: 12.77 minutes
    Epoch: 125/299 | Loss: 0.0015 | Episodes: 20 | Win count: 28 | Win rate: 0.875 | time: 12.81 minutes
    Epoch: 126/299 | Loss: 0.0019 | Episodes: 32 | Win count: 29 | Win rate: 0.906 | time: 12.88 minutes
    Epoch: 127/299 | Loss: 0.0022 | Episodes: 136 | Win count: 28 | Win rate: 0.875 | time: 13.14 minutes
    Epoch: 128/299 | Loss: 0.0010 | Episodes: 132 | Win count: 27 | Win rate: 0.844 | time: 13.39 minutes
    Epoch: 129/299 | Loss: 0.0007 | Episodes: 7 | Win count: 27 | Win rate: 0.844 | time: 13.40 minutes
    Epoch: 130/299 | Loss: 0.0012 | Episodes: 142 | Win count: 27 | Win rate: 0.844 | time: 13.67 minutes
    Epoch: 131/299 | Loss: 0.0011 | Episodes: 134 | Win count: 27 | Win rate: 0.844 | time: 13.92 minutes
    Epoch: 132/299 | Loss: 0.0014 | Episodes: 54 | Win count: 27 | Win rate: 0.844 | time: 14.03 minutes
    Epoch: 133/299 | Loss: 0.0009 | Episodes: 101 | Win count: 27 | Win rate: 0.844 | time: 14.22 minutes
    Epoch: 134/299 | Loss: 0.0012 | Episodes: 8 | Win count: 27 | Win rate: 0.844 | time: 14.23 minutes
    Epoch: 135/299 | Loss: 0.0014 | Episodes: 40 | Win count: 27 | Win rate: 0.844 | time: 14.31 minutes
    Epoch: 136/299 | Loss: 0.0008 | Episodes: 15 | Win count: 27 | Win rate: 0.844 | time: 14.34 minutes
    Epoch: 137/299 | Loss: 0.0016 | Episodes: 109 | Win count: 28 | Win rate: 0.875 | time: 14.55 minutes
    Epoch: 138/299 | Loss: 0.0004 | Episodes: 87 | Win count: 28 | Win rate: 0.875 | time: 14.72 minutes
    Epoch: 139/299 | Loss: 0.0032 | Episodes: 27 | Win count: 28 | Win rate: 0.875 | time: 14.77 minutes
    Epoch: 140/299 | Loss: 0.0014 | Episodes: 126 | Win count: 28 | Win rate: 0.875 | time: 15.02 minutes
    Epoch: 141/299 | Loss: 0.0036 | Episodes: 17 | Win count: 28 | Win rate: 0.875 | time: 15.05 minutes
    Epoch: 142/299 | Loss: 0.0016 | Episodes: 84 | Win count: 28 | Win rate: 0.875 | time: 15.21 minutes
    Epoch: 143/299 | Loss: 0.0012 | Episodes: 71 | Win count: 28 | Win rate: 0.875 | time: 15.34 minutes
    Epoch: 144/299 | Loss: 0.0012 | Episodes: 6 | Win count: 28 | Win rate: 0.875 | time: 15.36 minutes
    Epoch: 145/299 | Loss: 0.0010 | Episodes: 139 | Win count: 27 | Win rate: 0.844 | time: 15.62 minutes
    Epoch: 146/299 | Loss: 0.0005 | Episodes: 150 | Win count: 26 | Win rate: 0.812 | time: 15.91 minutes
    Epoch: 147/299 | Loss: 0.0009 | Episodes: 24 | Win count: 26 | Win rate: 0.812 | time: 15.96 minutes
    Epoch: 148/299 | Loss: 0.0008 | Episodes: 139 | Win count: 25 | Win rate: 0.781 | time: 16.24 minutes
    Epoch: 149/299 | Loss: 0.0020 | Episodes: 131 | Win count: 24 | Win rate: 0.750 | time: 16.49 minutes
    Epoch: 150/299 | Loss: 0.0001 | Episodes: 19 | Win count: 24 | Win rate: 0.750 | time: 16.52 minutes
    Epoch: 151/299 | Loss: 0.0042 | Episodes: 134 | Win count: 23 | Win rate: 0.719 | time: 16.78 minutes
    Epoch: 152/299 | Loss: 0.0002 | Episodes: 69 | Win count: 23 | Win rate: 0.719 | time: 16.91 minutes
    Epoch: 153/299 | Loss: 0.0001 | Episodes: 140 | Win count: 22 | Win rate: 0.688 | time: 17.18 minutes
    Epoch: 154/299 | Loss: 0.0008 | Episodes: 135 | Win count: 21 | Win rate: 0.656 | time: 17.45 minutes
    Epoch: 155/299 | Loss: 0.0006 | Episodes: 1 | Win count: 21 | Win rate: 0.656 | time: 17.45 minutes
    Epoch: 156/299 | Loss: 0.0002 | Episodes: 6 | Win count: 22 | Win rate: 0.688 | time: 17.46 minutes
    Epoch: 157/299 | Loss: 0.0010 | Episodes: 11 | Win count: 22 | Win rate: 0.688 | time: 17.48 minutes
    Epoch: 158/299 | Loss: 0.0026 | Episodes: 130 | Win count: 22 | Win rate: 0.688 | time: 17.73 minutes
    Epoch: 159/299 | Loss: 0.0004 | Episodes: 54 | Win count: 23 | Win rate: 0.719 | time: 17.84 minutes
    Epoch: 160/299 | Loss: 0.0008 | Episodes: 98 | Win count: 24 | Win rate: 0.750 | time: 18.03 minutes
    Epoch: 161/299 | Loss: 0.0017 | Episodes: 21 | Win count: 24 | Win rate: 0.750 | time: 18.07 minutes
    Epoch: 162/299 | Loss: 0.0015 | Episodes: 23 | Win count: 24 | Win rate: 0.750 | time: 18.12 minutes
    Epoch: 163/299 | Loss: 0.0010 | Episodes: 15 | Win count: 25 | Win rate: 0.781 | time: 18.15 minutes
    Epoch: 164/299 | Loss: 0.0012 | Episodes: 14 | Win count: 25 | Win rate: 0.781 | time: 18.17 minutes
    Epoch: 165/299 | Loss: 0.0008 | Episodes: 132 | Win count: 24 | Win rate: 0.750 | time: 18.42 minutes
    Epoch: 166/299 | Loss: 0.0016 | Episodes: 6 | Win count: 24 | Win rate: 0.750 | time: 18.43 minutes
    Epoch: 167/299 | Loss: 0.0002 | Episodes: 136 | Win count: 23 | Win rate: 0.719 | time: 18.70 minutes
    Epoch: 168/299 | Loss: 0.0009 | Episodes: 141 | Win count: 22 | Win rate: 0.688 | time: 18.97 minutes
    Epoch: 169/299 | Loss: 0.0002 | Episodes: 142 | Win count: 21 | Win rate: 0.656 | time: 19.25 minutes
    Epoch: 170/299 | Loss: 0.0010 | Episodes: 130 | Win count: 20 | Win rate: 0.625 | time: 19.50 minutes
    Epoch: 171/299 | Loss: 0.0000 | Episodes: 137 | Win count: 19 | Win rate: 0.594 | time: 19.76 minutes
    Epoch: 172/299 | Loss: 0.0002 | Episodes: 139 | Win count: 18 | Win rate: 0.562 | time: 20.03 minutes
    Epoch: 173/299 | Loss: 0.0003 | Episodes: 1 | Win count: 18 | Win rate: 0.562 | time: 20.03 minutes
    Epoch: 174/299 | Loss: 0.0011 | Episodes: 104 | Win count: 18 | Win rate: 0.562 | time: 20.23 minutes
    Epoch: 175/299 | Loss: 0.0004 | Episodes: 3 | Win count: 18 | Win rate: 0.562 | time: 20.24 minutes
    Epoch: 176/299 | Loss: 0.0003 | Episodes: 138 | Win count: 17 | Win rate: 0.531 | time: 20.51 minutes
    Epoch: 177/299 | Loss: 0.0006 | Episodes: 11 | Win count: 18 | Win rate: 0.562 | time: 20.53 minutes
    Epoch: 178/299 | Loss: 0.0002 | Episodes: 10 | Win count: 19 | Win rate: 0.594 | time: 20.55 minutes
    Epoch: 179/299 | Loss: 0.0008 | Episodes: 124 | Win count: 19 | Win rate: 0.594 | time: 20.79 minutes
    Epoch: 180/299 | Loss: 0.0002 | Episodes: 1 | Win count: 20 | Win rate: 0.625 | time: 20.80 minutes
    Epoch: 181/299 | Loss: 0.0102 | Episodes: 1 | Win count: 21 | Win rate: 0.656 | time: 20.80 minutes
    Epoch: 182/299 | Loss: 0.0032 | Episodes: 4 | Win count: 21 | Win rate: 0.656 | time: 20.81 minutes
    Epoch: 183/299 | Loss: 0.0013 | Episodes: 143 | Win count: 21 | Win rate: 0.656 | time: 21.08 minutes
    Epoch: 184/299 | Loss: 0.0014 | Episodes: 27 | Win count: 21 | Win rate: 0.656 | time: 21.14 minutes
    Epoch: 185/299 | Loss: 0.0004 | Episodes: 5 | Win count: 22 | Win rate: 0.688 | time: 21.15 minutes
    Epoch: 186/299 | Loss: 0.0011 | Episodes: 9 | Win count: 23 | Win rate: 0.719 | time: 21.16 minutes
    Epoch: 187/299 | Loss: 0.0013 | Episodes: 139 | Win count: 22 | Win rate: 0.688 | time: 21.43 minutes
    Epoch: 188/299 | Loss: 0.0009 | Episodes: 8 | Win count: 22 | Win rate: 0.688 | time: 21.44 minutes
    Epoch: 189/299 | Loss: 0.0024 | Episodes: 13 | Win count: 22 | Win rate: 0.688 | time: 21.47 minutes
    Epoch: 190/299 | Loss: 0.0052 | Episodes: 5 | Win count: 22 | Win rate: 0.688 | time: 21.48 minutes
    Epoch: 191/299 | Loss: 0.0014 | Episodes: 16 | Win count: 22 | Win rate: 0.688 | time: 21.51 minutes
    Epoch: 192/299 | Loss: 0.0016 | Episodes: 1 | Win count: 22 | Win rate: 0.688 | time: 21.51 minutes
    Epoch: 193/299 | Loss: 0.0012 | Episodes: 12 | Win count: 22 | Win rate: 0.688 | time: 21.54 minutes
    Epoch: 194/299 | Loss: 0.0008 | Episodes: 9 | Win count: 22 | Win rate: 0.688 | time: 21.55 minutes
    Epoch: 195/299 | Loss: 0.0003 | Episodes: 137 | Win count: 21 | Win rate: 0.656 | time: 21.82 minutes
    Epoch: 196/299 | Loss: 0.0006 | Episodes: 139 | Win count: 20 | Win rate: 0.625 | time: 22.09 minutes
    Epoch: 197/299 | Loss: 0.0008 | Episodes: 134 | Win count: 20 | Win rate: 0.625 | time: 22.35 minutes
    Epoch: 198/299 | Loss: 0.0007 | Episodes: 134 | Win count: 19 | Win rate: 0.594 | time: 22.61 minutes
    Epoch: 199/299 | Loss: 0.0006 | Episodes: 11 | Win count: 20 | Win rate: 0.625 | time: 22.63 minutes
    Epoch: 200/299 | Loss: 0.0002 | Episodes: 139 | Win count: 20 | Win rate: 0.625 | time: 22.89 minutes
    Epoch: 201/299 | Loss: 0.0003 | Episodes: 134 | Win count: 20 | Win rate: 0.625 | time: 23.15 minutes
    Epoch: 202/299 | Loss: 0.0002 | Episodes: 138 | Win count: 20 | Win rate: 0.625 | time: 23.41 minutes
    Epoch: 203/299 | Loss: 0.0007 | Episodes: 2 | Win count: 21 | Win rate: 0.656 | time: 23.42 minutes
    Epoch: 204/299 | Loss: 0.0008 | Episodes: 134 | Win count: 21 | Win rate: 0.656 | time: 23.68 minutes
    Epoch: 205/299 | Loss: 0.0006 | Episodes: 139 | Win count: 20 | Win rate: 0.625 | time: 23.95 minutes
    Epoch: 206/299 | Loss: 0.0006 | Episodes: 2 | Win count: 20 | Win rate: 0.625 | time: 23.96 minutes
    Epoch: 207/299 | Loss: 0.0017 | Episodes: 17 | Win count: 20 | Win rate: 0.625 | time: 23.99 minutes
    Epoch: 208/299 | Loss: 0.0002 | Episodes: 133 | Win count: 20 | Win rate: 0.625 | time: 24.26 minutes
    Epoch: 209/299 | Loss: 0.0008 | Episodes: 9 | Win count: 20 | Win rate: 0.625 | time: 24.27 minutes
    Epoch: 210/299 | Loss: 0.0011 | Episodes: 4 | Win count: 20 | Win rate: 0.625 | time: 24.28 minutes
    Epoch: 211/299 | Loss: 0.0006 | Episodes: 4 | Win count: 20 | Win rate: 0.625 | time: 24.29 minutes
    Epoch: 212/299 | Loss: 0.0014 | Episodes: 10 | Win count: 20 | Win rate: 0.625 | time: 24.31 minutes
    Epoch: 213/299 | Loss: 0.0014 | Episodes: 143 | Win count: 19 | Win rate: 0.594 | time: 24.58 minutes
    Epoch: 214/299 | Loss: 0.0008 | Episodes: 9 | Win count: 19 | Win rate: 0.594 | time: 24.60 minutes
    Epoch: 215/299 | Loss: 0.0010 | Episodes: 133 | Win count: 19 | Win rate: 0.594 | time: 24.85 minutes
    Epoch: 216/299 | Loss: 0.0003 | Episodes: 134 | Win count: 18 | Win rate: 0.562 | time: 25.12 minutes
    Epoch: 217/299 | Loss: 0.0006 | Episodes: 135 | Win count: 17 | Win rate: 0.531 | time: 25.38 minutes
    Epoch: 218/299 | Loss: 0.0006 | Episodes: 135 | Win count: 16 | Win rate: 0.500 | time: 25.65 minutes
    Epoch: 219/299 | Loss: 0.0006 | Episodes: 132 | Win count: 16 | Win rate: 0.500 | time: 25.90 minutes
    Epoch: 220/299 | Loss: 0.0014 | Episodes: 52 | Win count: 16 | Win rate: 0.500 | time: 26.00 minutes
    Epoch: 221/299 | Loss: 0.0016 | Episodes: 137 | Win count: 15 | Win rate: 0.469 | time: 26.26 minutes
    Epoch: 222/299 | Loss: 0.0008 | Episodes: 5 | Win count: 15 | Win rate: 0.469 | time: 26.27 minutes
    Epoch: 223/299 | Loss: 0.0008 | Episodes: 61 | Win count: 15 | Win rate: 0.469 | time: 26.39 minutes
    Epoch: 224/299 | Loss: 0.0005 | Episodes: 136 | Win count: 14 | Win rate: 0.438 | time: 26.65 minutes
    Epoch: 225/299 | Loss: 0.0009 | Episodes: 44 | Win count: 14 | Win rate: 0.438 | time: 26.74 minutes
    Epoch: 226/299 | Loss: 0.0005 | Episodes: 9 | Win count: 14 | Win rate: 0.438 | time: 26.75 minutes
    Epoch: 227/299 | Loss: 0.0005 | Episodes: 10 | Win count: 15 | Win rate: 0.469 | time: 26.77 minutes
    Epoch: 228/299 | Loss: 0.0011 | Episodes: 2 | Win count: 16 | Win rate: 0.500 | time: 26.78 minutes
    Epoch: 229/299 | Loss: 0.0004 | Episodes: 88 | Win count: 17 | Win rate: 0.531 | time: 26.95 minutes
    Epoch: 230/299 | Loss: 0.0001 | Episodes: 136 | Win count: 17 | Win rate: 0.531 | time: 27.21 minutes
    Epoch: 231/299 | Loss: 0.0005 | Episodes: 134 | Win count: 16 | Win rate: 0.500 | time: 27.47 minutes
    Epoch: 232/299 | Loss: 0.0008 | Episodes: 135 | Win count: 16 | Win rate: 0.500 | time: 27.73 minutes
    Epoch: 233/299 | Loss: 0.0010 | Episodes: 57 | Win count: 17 | Win rate: 0.531 | time: 27.83 minutes
    Epoch: 234/299 | Loss: 0.0010 | Episodes: 131 | Win count: 17 | Win rate: 0.531 | time: 28.09 minutes
    Epoch: 235/299 | Loss: 0.0005 | Episodes: 132 | Win count: 16 | Win rate: 0.500 | time: 28.35 minutes
    Epoch: 236/299 | Loss: 0.0003 | Episodes: 131 | Win count: 16 | Win rate: 0.500 | time: 28.60 minutes
    Epoch: 237/299 | Loss: 0.0003 | Episodes: 134 | Win count: 16 | Win rate: 0.500 | time: 28.86 minutes
    Epoch: 238/299 | Loss: 0.0003 | Episodes: 139 | Win count: 15 | Win rate: 0.469 | time: 29.13 minutes
    Epoch: 239/299 | Loss: 0.0001 | Episodes: 133 | Win count: 14 | Win rate: 0.438 | time: 29.38 minutes
    Epoch: 240/299 | Loss: 0.0002 | Episodes: 141 | Win count: 14 | Win rate: 0.438 | time: 29.66 minutes
    Epoch: 241/299 | Loss: 0.0052 | Episodes: 133 | Win count: 13 | Win rate: 0.406 | time: 29.92 minutes
    Epoch: 242/299 | Loss: 0.0151 | Episodes: 49 | Win count: 13 | Win rate: 0.406 | time: 30.01 minutes
    Epoch: 243/299 | Loss: 0.0014 | Episodes: 17 | Win count: 13 | Win rate: 0.406 | time: 30.05 minutes
    Epoch: 244/299 | Loss: 0.0021 | Episodes: 136 | Win count: 12 | Win rate: 0.375 | time: 30.31 minutes
    Epoch: 245/299 | Loss: 0.0057 | Episodes: 7 | Win count: 13 | Win rate: 0.406 | time: 30.32 minutes
    Epoch: 246/299 | Loss: 0.0066 | Episodes: 19 | Win count: 13 | Win rate: 0.406 | time: 30.36 minutes
    Epoch: 247/299 | Loss: 0.0030 | Episodes: 3 | Win count: 14 | Win rate: 0.438 | time: 30.36 minutes
    Epoch: 248/299 | Loss: 0.0025 | Episodes: 12 | Win count: 15 | Win rate: 0.469 | time: 30.39 minutes
    Epoch: 249/299 | Loss: 0.0060 | Episodes: 10 | Win count: 16 | Win rate: 0.500 | time: 30.41 minutes
    Epoch: 250/299 | Loss: 0.0048 | Episodes: 60 | Win count: 17 | Win rate: 0.531 | time: 30.52 minutes
    Epoch: 251/299 | Loss: 0.0014 | Episodes: 51 | Win count: 18 | Win rate: 0.562 | time: 30.62 minutes
    Epoch: 252/299 | Loss: 0.0014 | Episodes: 89 | Win count: 18 | Win rate: 0.562 | time: 30.79 minutes
    Epoch: 253/299 | Loss: 0.0018 | Episodes: 65 | Win count: 19 | Win rate: 0.594 | time: 30.91 minutes
    Epoch: 254/299 | Loss: 0.0009 | Episodes: 8 | Win count: 19 | Win rate: 0.594 | time: 30.92 minutes
    Epoch: 255/299 | Loss: 0.0008 | Episodes: 65 | Win count: 19 | Win rate: 0.594 | time: 31.05 minutes
    Epoch: 256/299 | Loss: 0.0009 | Episodes: 17 | Win count: 20 | Win rate: 0.625 | time: 31.08 minutes
    Epoch: 257/299 | Loss: 0.0029 | Episodes: 11 | Win count: 20 | Win rate: 0.625 | time: 31.10 minutes
    Epoch: 258/299 | Loss: 0.0027 | Episodes: 6 | Win count: 20 | Win rate: 0.625 | time: 31.11 minutes
    Epoch: 259/299 | Loss: 0.0019 | Episodes: 2 | Win count: 20 | Win rate: 0.625 | time: 31.12 minutes
    Epoch: 260/299 | Loss: 0.0012 | Episodes: 16 | Win count: 20 | Win rate: 0.625 | time: 31.15 minutes
    Epoch: 261/299 | Loss: 0.0044 | Episodes: 21 | Win count: 20 | Win rate: 0.625 | time: 31.19 minutes
    Epoch: 262/299 | Loss: 0.0115 | Episodes: 7 | Win count: 21 | Win rate: 0.656 | time: 31.21 minutes
    Epoch: 263/299 | Loss: 0.0015 | Episodes: 12 | Win count: 22 | Win rate: 0.688 | time: 31.23 minutes
    Epoch: 264/299 | Loss: 0.0023 | Episodes: 1 | Win count: 23 | Win rate: 0.719 | time: 31.23 minutes
    Epoch: 265/299 | Loss: 0.0018 | Episodes: 33 | Win count: 23 | Win rate: 0.719 | time: 31.30 minutes
    Epoch: 266/299 | Loss: 0.0008 | Episodes: 9 | Win count: 24 | Win rate: 0.750 | time: 31.32 minutes
    Epoch: 267/299 | Loss: 0.0017 | Episodes: 131 | Win count: 24 | Win rate: 0.750 | time: 31.57 minutes
    Epoch: 268/299 | Loss: 0.0029 | Episodes: 7 | Win count: 25 | Win rate: 0.781 | time: 31.59 minutes
    Epoch: 269/299 | Loss: 0.0005 | Episodes: 7 | Win count: 26 | Win rate: 0.812 | time: 31.60 minutes
    Epoch: 270/299 | Loss: 0.0015 | Episodes: 131 | Win count: 26 | Win rate: 0.812 | time: 31.86 minutes
    Epoch: 271/299 | Loss: 0.0022 | Episodes: 38 | Win count: 27 | Win rate: 0.844 | time: 31.93 minutes
    Epoch: 272/299 | Loss: 0.0015 | Episodes: 8 | Win count: 28 | Win rate: 0.875 | time: 31.94 minutes
    Epoch: 273/299 | Loss: 0.0011 | Episodes: 35 | Win count: 29 | Win rate: 0.906 | time: 32.01 minutes
    Epoch: 274/299 | Loss: 0.0009 | Episodes: 134 | Win count: 28 | Win rate: 0.875 | time: 32.27 minutes
    Epoch: 275/299 | Loss: 0.0007 | Episodes: 132 | Win count: 27 | Win rate: 0.844 | time: 32.52 minutes
    Epoch: 276/299 | Loss: 0.0013 | Episodes: 29 | Win count: 28 | Win rate: 0.875 | time: 32.57 minutes
    Epoch: 277/299 | Loss: 0.0042 | Episodes: 133 | Win count: 27 | Win rate: 0.844 | time: 32.83 minutes
    Epoch: 278/299 | Loss: 0.0000 | Episodes: 131 | Win count: 26 | Win rate: 0.812 | time: 33.09 minutes
    Epoch: 279/299 | Loss: 0.0011 | Episodes: 134 | Win count: 25 | Win rate: 0.781 | time: 33.35 minutes
    Epoch: 280/299 | Loss: 0.0004 | Episodes: 132 | Win count: 24 | Win rate: 0.750 | time: 33.61 minutes
    Epoch: 281/299 | Loss: 0.0019 | Episodes: 10 | Win count: 24 | Win rate: 0.750 | time: 33.62 minutes
    Epoch: 282/299 | Loss: 0.0006 | Episodes: 9 | Win count: 24 | Win rate: 0.750 | time: 33.64 minutes
    Epoch: 283/299 | Loss: 0.0007 | Episodes: 134 | Win count: 23 | Win rate: 0.719 | time: 33.90 minutes
    Epoch: 284/299 | Loss: 0.0014 | Episodes: 28 | Win count: 23 | Win rate: 0.719 | time: 33.95 minutes
    Epoch: 285/299 | Loss: 0.0061 | Episodes: 132 | Win count: 22 | Win rate: 0.688 | time: 34.20 minutes
    Epoch: 286/299 | Loss: 0.0005 | Episodes: 8 | Win count: 22 | Win rate: 0.688 | time: 34.22 minutes
    Epoch: 287/299 | Loss: 0.0003 | Episodes: 2 | Win count: 22 | Win rate: 0.688 | time: 34.22 minutes
    Epoch: 288/299 | Loss: 0.0006 | Episodes: 138 | Win count: 21 | Win rate: 0.656 | time: 34.49 minutes
    Epoch: 289/299 | Loss: 0.0015 | Episodes: 10 | Win count: 21 | Win rate: 0.656 | time: 34.51 minutes
    Epoch: 290/299 | Loss: 0.0003 | Episodes: 1 | Win count: 21 | Win rate: 0.656 | time: 34.51 minutes
    Epoch: 291/299 | Loss: 0.0008 | Episodes: 13 | Win count: 21 | Win rate: 0.656 | time: 34.54 minutes
    Epoch: 292/299 | Loss: 0.0007 | Episodes: 15 | Win count: 21 | Win rate: 0.656 | time: 34.57 minutes
    Epoch: 293/299 | Loss: 0.0010 | Episodes: 135 | Win count: 20 | Win rate: 0.625 | time: 34.83 minutes
    Epoch: 294/299 | Loss: 0.0003 | Episodes: 2 | Win count: 20 | Win rate: 0.625 | time: 34.83 minutes
    Epoch: 295/299 | Loss: 0.0011 | Episodes: 53 | Win count: 20 | Win rate: 0.625 | time: 34.93 minutes
    Epoch: 296/299 | Loss: 0.0001 | Episodes: 21 | Win count: 20 | Win rate: 0.625 | time: 34.97 minutes
    Epoch: 297/299 | Loss: 0.0001 | Episodes: 12 | Win count: 20 | Win rate: 0.625 | time: 34.99 minutes
    Epoch: 298/299 | Loss: 0.0007 | Episodes: 131 | Win count: 19 | Win rate: 0.594 | time: 35.24 minutes
    Epoch: 299/299 | Loss: 0.0004 | Episodes: 14 | Win count: 20 | Win rate: 0.625 | time: 35.26 minutes
    n_epoch: 300, max_memory: 512, data_size: 32, time: 35.26 minutes

## Test the trained model<a href="#Test-the-trained-model" class="anchor-link"
rel="noopener">¶</a>

In \[8\]:

    qmaze = TreasureMaze(maze)
    pirate_cell = (0, 0)
    win = play_game(model, qmaze, pirate_cell)
    print("Pirate won:", win)
    show(qmaze)

    Pirate won: False

Out\[8\]:

    <matplotlib.image.AxesImage at 0x75b7a812cb90>

![No description has been provided for this image](javascript://)

## Submission note<a href="#Submission-note" class="anchor-link" rel="noopener">¶</a>

Export this notebook as **HTML** for submission. Submit the design
defense as a separate Word document in APA style.

In \[ \]:

     

In \[ \]:

     

In \[ \]: